# Tugas 6 :  Membuat pencarian Query dengan kalimat dalam dokumen dengan penerapan Singular Value Decomposition (SVD)

NAMA : Mohammad Hasan Basri

NIM  : 210411100169

MATA KULIAH : Pencarian dan Penambangan Web - A

mencari 100 dokumen itu (masukkan 1 kalimat, ketemunya dokumen yg mana)

data tfidf ditransofrmasi menjadi sdikit dimensi menggunakan svd
yg diambil v dan sigma (v tdk diambil smua)(Sebagian dri v)
mentransformasi supaya dikit kolom


cosinus similtas (mirip mendekati 1) boleh pke euclidean

- membuat sistem pencarian dokumen
- reduksi dimensi SVD
- data baru juga di reduksi
- mencari kemiripan ecludian distance

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd

data_path = ("/content/drive/My Drive/PPWA/report/Tugas_PPWA/Hasil_Preprocessing_Hasan.csv")
news_data = pd.read_csv(data_path)

news_data

,judul,tanggal,isi,kategori,cleansing,case_folding,tokenize,stopword_removal
0,Iran Serukan Negara Muslim Bersatu Hentikan Ke...,"Kamis, 17 Okt 2024 10:01 WIB",Jakarta - Presiden Iran Masoud Pezeshkian mene...,Keislaman,Jakarta Presiden Iran Masoud Pezeshkian menek...,jakarta presiden iran masoud pezeshkian menek...,"['jakarta', 'presiden', 'iran', 'masoud', 'pez...",jakarta presiden iran masoud pezeshkian meneka...
1,"Bacaan Al-Qur'an Surah Asy-Syu'ara: Arab, Lati...","Kamis, 17 Okt 2024 09:30 WIB",NaN,Keislaman,NaN,NaN,['nan'],NaN
2,"Rukun Iman Ke-2, Setiap Muslim Wajib Mengimani...","Kamis, 17 Okt 2024 08:45 WIB",Jakarta - Malaikat adalah makhluk ciptaan Alla...,Keislaman,Jakarta Malaikat adalah makhluk ciptaan Allah...,jakarta malaikat adalah makhluk ciptaan allah...,"['jakarta', 'malaikat', 'adalah', 'makhluk', '...",jakarta malaikat makhluk ciptaan allah swt mus...
3,Kalender Ramadhan 2025 Muhammadiyah dan Predik...,"Kamis, 17 Okt 2024 08:00 WIB",Jakarta - Kalender Ramadhan 2025 versi Muhamma...,Keislaman,Jakarta Kalender Ramadhan versi Muhammadiyah...,jakarta kalender ramadhan versi muhammadiyah...,"['jakarta', 'kalender', 'ramadhan', 'versi', '...",jakarta kalender ramadhan versi muhammadiyah t...
4,20 Gambaran Kehidupan di Neraka Menurut Al-Qur...,"Kamis, 17 Okt 2024 07:15 WIB",Jakarta - Gambaran kehidupan di neraka selalu ...,Keislaman,Jakarta Gambaran kehidupan di neraka selalu m...,jakarta gambaran kehidupan di neraka selalu m...,"['jakarta', 'gambaran', 'kehidupan', 'di', 'ne...",jakarta gambaran kehidupan neraka pengingat ke...
...,...,...,...,...,...,...,...,...
95,"Praha Incar Turis-Turis Kaya, 'Wisata Pub Jala...","Rabu, 16 Okt 2024 06:13 WIB",Praha - Praha ingin mengganti imej dari kota s...,Pariwisata,Praha Praha ingin mengganti imej dari kota se...,praha praha ingin mengganti imej dari kota se...,"['praha', 'praha', 'ingin', 'mengganti', 'imej...",praha praha mengganti imej kota tujuan wisata ...
96,"Usai Festival Vegetarian, Phuket Pusing dengan...","Rabu, 16 Okt 2024 05:39 WIB",Phuket - Festival Vegetarian menjadi salah sat...,Pariwisata,Phuket Festival Vegetarian menjadi salah satu...,phuket festival vegetarian menjadi salah satu...,"['phuket', 'festival', 'vegetarian', 'menjadi'...",phuket festival vegetarian salah daya tarik pa...
97,Ngeri! Turis Inggris Tewas Saat Memanjat Jemba...,"Rabu, 16 Okt 2024 05:01 WIB",Jakarta - Seorang turis yang juga kreator kont...,Pariwisata,Jakarta Seorang turis yang juga kreator konte...,jakarta seorang turis yang juga kreator konte...,"['jakarta', 'seorang', 'turis', 'yang', 'juga'...",jakarta turis kreator konten inggris berusia t...
98,Info Loker: Lion Group Buka Lowongan Mekanik Nih!,"Selasa, 15 Okt 2024 23:04 WIB",Jakarta - Traveler yang ingin bekerja di dunia...,Pariwisata,Jakarta Traveler yang ingin bekerja di dunia ...,jakarta traveler yang ingin bekerja di dunia ...,"['jakarta', 'traveler', 'yang', 'ingin', 'beke...",jakarta traveler dunia aviasi baiknya simak in...


In [3]:
# Ambil kolom dokumen berita
documents = news_data['stopword_removal'].tolist()

**TF-IDF (Term Frequency-Inverse Document Frequency)**

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import euclidean_distances
import pandas as pd
import numpy as np
import joblib

*   TfidfVectorizer adalah alat dari pustaka scikit-learn untuk mengubah kumpulan teks menjadi representasi vektor berbasis Term Frequency-Inverse Document Frequency (TF-IDF)

*   from sklearn.feature_extraction.text adalah modul dalam pustaka scikit-learn yang menyediakan berbagai alat untuk mengekstraksi fitur atau informasi numerik dari data teks (seperti dokumen atau kalimat) agar bisa digunakan dalam model pembelajaran mesin. Salah satu alat penting yang disediakan modul ini adalah TfidfVectorizer, yang digunakan untuk mengonversi teks menjadi representasi numerik berbasis frekuensi.

*   from sklearn.decomposition import TruncatedSVD adalah bagian dari pustaka scikit-learn yang menyediakan alat untuk melakukan reduksi dimensi pada data numerik, terutama matriks besar, dengan menggunakan teknik Truncated Singular Value Decomposition (Truncated SVD). Truncated SVD biasanya diterapkan pada data yang telah diproses menjadi bentuk matriks dengan representasi TF-IDF atau lainnya, terutama pada data teks untuk analisis Latent Semantic Analysis (LSA).

*   Truncated SVD adalah varian dari Singular Value Decomposition (SVD), yang menguraikan matriks menjadi tiga matriks komponen, tetapi dengan hanya menyimpan komponen terbesar. Dalam konteks pembelajaran mesin dan pemrosesan data teks, tujuan utamanya adalah untuk mereduksi dimensi data agar menjadi lebih sederhana, namun tetap mempertahankan informasi penting.

*   from sklearn.metrics.pairwise import euclidean_distances adalah bagian dari pustaka scikit-learn yang digunakan untuk menghitung jarak Euclidean antara pasangan data dalam matriks. Jarak Euclidean adalah ukuran jarak "lurus" antara dua titik dalam ruang dimensi. Di dalam konteks pemrosesan teks atau data numerik, ini sering digunakan untuk mengukur seberapa dekat atau jauh dua dokumen atau dua sampel dalam ruang vektor.

*   euclidean_distances digunakan untuk menghitung jarak Euclidean antara setiap pasangan baris dalam dua matriks. Jika hanya ada satu matriks yang diberikan, euclidean_distances akan menghitung jarak Euclidean antara semua baris dalam matriks tersebut, sehingga menghasilkan matriks jarak.

Matriks jarak Euclidean menunjukkan jarak antara setiap dokumen:

*   Nilai yang lebih kecil mengindikasikan bahwa dua dokumen tersebut lebih mirip secara semantik.

*   Nilai yang lebih besar menunjukkan bahwa dua dokumen memiliki perbedaan yang lebih jauh.

*   pandas adalah pustaka Python yang digunakan untuk manipulasi dan analisis data. Dengan menggunakan pandas, Anda dapat dengan mudah bekerja dengan data yang terstruktur dalam bentuk tabel (DataFrame) serta melakukan operasi seperti pemfilteran, pengelompokan, agregasi, dan transformasi data.

*   numpy adalah pustaka Python yang digunakan untuk komputasi numerik. Pustaka ini menyediakan dukungan untuk array multidimensi dan berbagai fungsi matematis yang efisien untuk memanipulasi data tersebut.

*   Digunakan untuk menyimpan dan memuat objek Python, joblib adalah pustaka Python yang digunakan untuk serialisasi (menyimpan) dan deserialisasi (memuat) objek Python, khususnya yang besar, seperti model machine learning.















In [5]:
# tfidf_vectorizer = TfidfVectorizer()
# tfidf_matrix = tfidf_vectorizer.fit_transform(documents)

# # Mendapatkan fitur (kata-kata) dari TF-IDF
# tfidf_features = tfidf_vectorizer.get_feature_names_out()

# # Mengonversi matriks TF-IDF menjadi DataFrame
# tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=tfidf_features)

# # Tampilkan hasil TF-IDF
# print("\nHasil TF-IDF dalam bentuk DataFrame:")
# tfidf_df

In [9]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd

# Replace np.nan values in 'documents' with empty strings
documents = [str(doc) if not pd.isna(doc) else '' for doc in documents]

# Misalkan 'documents' adalah daftar dokumen yang ingin dianalisis
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(documents)

# Mendapatkan fitur (kata-kata) dari TF-IDF
tfidf_features = tfidf_vectorizer.get_feature_names_out()

# Mengonversi matriks TF-IDF menjadi DataFrame dan mentransposenya
tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=tfidf_features).T

# Tampilkan hasil TF-IDF
print("\nHasil TF-IDF dalam bentuk DataFrame (fitur sebagai baris):")
print(tfidf_df)


Hasil TF-IDF dalam bentuk DataFrame (fitur sebagai baris):
            0    1    2         3    4         5    6    7    8    9   ...  \
aamiin     0.0  0.0  0.0  0.000000  0.0  0.000000  0.0  0.0  0.0  0.0  ...   
ab         0.0  0.0  0.0  0.000000  0.0  0.000000  0.0  0.0  0.0  0.0  ...   
abaa       0.0  0.0  0.0  0.000000  0.0  0.000000  0.0  0.0  0.0  0.0  ...   
abaabiil   0.0  0.0  0.0  0.000000  0.0  0.000000  0.0  0.0  0.0  0.0  ...   
abaabiila  0.0  0.0  0.0  0.000000  0.0  0.000000  0.0  0.0  0.0  0.0  ...   
...        ...  ...  ...       ...  ...       ...  ...  ...  ...  ...  ...   
zubkov     0.0  0.0  0.0  0.000000  0.0  0.000000  0.0  0.0  0.0  0.0  ...   
zulaikha   0.0  0.0  0.0  0.000000  0.0  0.000000  0.0  0.0  0.0  0.0  ...   
zulfi      0.0  0.0  0.0  0.000000  0.0  0.043141  0.0  0.0  0.0  0.0  ...   
zulhijah   0.0  0.0  0.0  0.016557  0.0  0.000000  0.0  0.0  0.0  0.0  ...   
zulkipli   0.0  0.0  0.0  0.000000  0.0  0.000000  0.0  0.0  0.0  0.0  ...   

   

Menggunakan .T saat membuat tfidf_df, sehingga hasil akhirnya adalah fitur (kata-kata) sebagai baris dan setiap kolom mewakili dokumen. Ini membuat hasil TF-IDF lebih mudah dibaca jika kita ingin menganalisis bobot setiap fitur (kata) per dokumen.


1.   membuat sebuah objek TfidfVectorizer. Ini adalah langkah pertama untuk mengonversi kumpulan dokumen teks menjadi representasi numerik berdasarkan frekuensi kata dengan mempertimbangkan seberapa umum atau jarang kata tersebut dalam koleksi dokumen.

2.   TfidfVectorizer mengubah teks menjadi matriks TF-IDF, yang merupakan representasi numerik dari kata-kata dalam dokumen.

3.   menggunakan metode fit_transform() pada objek tfidf_vectorizer untuk menerapkan TF-IDF ke kumpulan dokumen yang disimpan dalam variabel documents.

4.   Hasilnya, tfidf_matrix adalah matriks yang menyimpan nilai TF-IDF untuk setiap kata dalam setiap dokumen. Setiap baris mewakili satu dokumen, dan setiap kolom mewakili satu kata (fitur) dari kumpulan dokumen.

5.   Dengan menggunakan get_feature_names_out(), kita dapat mengambil daftar kata (fitur) yang telah diidentifikasi oleh TfidfVectorizer. Ini adalah kata-kata unik yang ada dalam kumpulan dokumen.

6.   Variabel tfidf_features sekarang berisi daftar semua kata yang digunakan dalam analisis TF-IDF.

7.   mengonversi matriks TF-IDF (yang awalnya dalam bentuk sparse matrix) menjadi DataFrame menggunakan pd.DataFrame().

8.   Metode toarray() digunakan untuk mengubah sparse matrix menjadi array NumPy, yang kemudian dapat dengan mudah dimasukkan ke dalam DataFrame.

9.   columns=tfidf_features memberikan nama kolom pada DataFrame sesuai dengan kata-kata (fitur) yang ada.

10.    menampilkan DataFrame tfidf_df yang berisi nilai TF-IDF untuk setiap kata di setiap dokumen.










**Singular Value Decomposition (SVD)**

In [ ]:
# # Menggunakan SVD untuk reduksi dimensi
# n_components = 100
# svd = TruncatedSVD(n_components=n_components)
# svd_matrix = svd.fit_transform(tfidf_matrix)

# # Mengubah hasil SVD menjadi DataFrame
# svd_df = pd.DataFrame(svd_matrix, columns=[f'feature {i+1}' for i in range(n_components)])

# # Menampilkan hasil
# print("Hasil reduksi dimensi dengan SVD:")
# svd_df

In [10]:
# Menggunakan SVD untuk reduksi dimensi
n_components = 100
svd = TruncatedSVD(n_components=n_components)
svd_matrix = svd.fit_transform(tfidf_matrix)

# Mengubah hasil SVD menjadi DataFrame
svd_df = pd.DataFrame(svd_matrix, columns=[f' {i+1}' for i in range(n_components)])

# Menampilkan hasil
print("Hasil reduksi dimensi dengan SVD:")
svd_df

Hasil reduksi dimensi dengan SVD:


,1,2,3,4,5,6,7,8,9,10,...,91,92,93,94,95,96,97,98,99,100
0,0.110493,0.001359,0.031066,0.001588,0.053761,0.036660,-0.029007,0.080733,-0.099305,0.042532,...,0.007694,0.006916,-0.004255,-0.002584,0.003955,-0.002007,0.002198,0.000414,0.000224,-2.987909e-19
1,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000e+00
2,0.282719,-0.205761,-0.029395,-0.030589,-0.036069,-0.014527,-0.002444,-0.017847,0.007204,-0.035738,...,0.002726,-0.000051,0.003010,-0.000772,-0.001028,-0.001149,-0.003963,-0.001884,-0.000308,-1.030060e-16
3,0.061380,0.023102,0.109489,0.080139,0.518540,-0.491117,0.202782,-0.013010,0.045813,-0.011109,...,0.039017,0.010242,0.002959,0.026359,-0.003785,-0.001713,0.002995,0.000739,-0.000012,-1.628675e-16
4,0.292447,-0.192141,-0.013284,-0.033423,-0.034398,-0.000757,0.011308,-0.008904,-0.017783,-0.090181,...,0.003751,0.004190,0.003915,-0.003198,0.003496,0.001719,-0.001416,-0.000133,-0.000229,2.161226e-16
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,0.062469,0.033656,0.081548,0.008528,0.064518,0.056404,-0.091251,0.011009,-0.092769,0.010968,...,0.011117,-0.013415,-0.001411,-0.000438,-0.000236,0.006870,0.007640,-0.000451,-0.000168,8.582016e-17
96,0.049629,0.024674,0.060501,0.013246,0.083287,0.062497,-0.100594,-0.021326,-0.045690,-0.006969,...,-0.008777,0.000129,0.001661,-0.000761,-0.003693,0.000670,0.002044,0.001853,0.000507,-1.072648e-17
97,0.108899,0.053186,0.046970,0.038515,0.085366,0.054746,-0.006478,0.102111,-0.172763,0.026996,...,0.002541,0.010726,0.001227,0.014159,0.010698,0.004199,-0.001766,0.007107,0.001844,-7.607764e-17
98,0.132533,0.149239,0.045425,-0.018484,0.027507,0.017035,-0.061102,0.016889,-0.177783,0.148279,...,0.015501,0.014468,0.016024,0.003582,-0.003636,-0.003106,-0.002070,0.001008,-0.001285,7.299899e-17


In [ ]:
# svd_df.head(100)

*   menggunakan SVD (Singular Value Decomposition) untuk reduksi dimensi pada matriks TF-IDF.

*   n_components: Di sini, kita menentukan jumlah komponen utama yang ingin kita ambil dari matriks TF-IDF. Dalam hal ini, kita ingin mengurangi dimensi ke 100 komponen. Jumlah ini bisa disesuaikan tergantung pada data dan tujuan analisis.

*   TruncatedSVD: Ini adalah kelas dari pustaka sklearn.decomposition yang digunakan untuk melakukan SVD dengan cara yang efisien pada data besar dan sparse (seperti matriks TF-IDF). TruncatedSVD melakukan reduksi dimensi dengan cara menyimpan informasi penting dari data dengan mengurangi jumlah dimensi.

*   fit_transform(): Metode ini melakukan dua langkah :

1.   Fit: Menyesuaikan model SVD ke matriks TF-IDF untuk menemukan struktur dan pola.

2.   Transform: Mengubah matriks TF-IDF menjadi bentuk baru (matriks SVD) dengan dimensi yang lebih rendah.

*   svd_matrix: Hasil dari fit_transform() adalah matriks baru yang memiliki dimensi n_samples x n_components. Setiap baris di svd_matrix merepresentasikan dokumen dalam ruang dimensi yang lebih rendah.

*   mengonversi hasil matriks SVD (yang merupakan array NumPy) menjadi sebuah DataFrame menggunakan pandas.

*   pd.DataFrame(svd_matrix, ...): Ini menciptakan DataFrame baru dengan baris yang sama dengan jumlah dokumen dan kolom yang sesuai dengan jumlah komponen yang telah ditentukan (100).

*   daftar komprehensif yang membuat nama kolom sebagai feature 1, feature 2, ..., hingga feature 100. Ini memberikan label pada kolom-kolom baru berdasarkan jumlah komponen.







**Implementasi**

In [11]:
import re
import string
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import nltk
nltk.download('stopwords')
nltk.download('punkt')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

*   import re: Memungkinkan kita menggunakan modul reguler ekspresi (regex) untuk pencarian dan manipulasi string. Ini berguna untuk membersihkan teks, seperti menghapus karakter yang tidak diinginkan.

*   import string: Digunakan untuk mengakses sekumpulan konstanta string, seperti daftar karakter alfabet, digit, dan tanda baca. Ini berguna untuk memfilter atau menghapus karakter tertentu dari teks.

*   from nltk.corpus import stopwords: Mengimpor daftar kata yang umum (stopwords) dari pustaka NLTK. Stopwords adalah kata-kata yang sering muncul dalam bahasa tetapi tidak memiliki makna penting untuk analisis, seperti "dan", "di", "yang", dll.

*   from nltk.tokenize import word_tokenize: Mengimpor fungsi untuk memisahkan (tokenisasi) teks menjadi kata-kata. Tokenisasi adalah langkah awal penting dalam pemrosesan teks.

*   import nltk: Mengimpor pustaka NLTK itu sendiri. Ini adalah pustaka yang sangat kuat untuk  menyediakan berbagai alat dan sumber daya.

*   nltk.download('stopwords'): Mengunduh daftar stopwords yang diperlukan dari repositori NLTK. Ini akan memungkinkan Anda menggunakan daftar kata yang umum dalam bahasa Inggris (atau bahasa lain) untuk membersihkan teks.

*   nltk.download('punkt'): Mengunduh tokenizer yang digunakan untuk memisahkan teks menjadi kata dan kalimat. punkt adalah model untuk memisahkan kalimat dan kata, yang dapat menangani berbagai jenis teks dengan baik.




In [14]:
import re
import string
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import nltk

# Download the necessary NLTK data
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')  # This line is added to download the punkt_tab data


def preprocess_text(text):
    # Cleaning: Menghapus angka dan tanda baca
    text = re.sub(r'\d+', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = text.lower()
    words = word_tokenize(text)

    # Stopword removal
    stop_words = set(stopwords.words("indonesian"))
    words = [word for word in words if word not in stop_words]

    return ' '.join(words)

def search_similar_documents(query_text, top_k=5):
    query_text = preprocess_text(query_text)

    # Transformasi query text
    query_tfidf = tfidf_vectorizer.transform([query_text])
    query_svd = svd.transform(query_tfidf)

    # Hitung jarak Euclidean antara query dan dokumen
    distances = euclidean_distances(query_svd, svd_matrix)

    # Ambil indeks dokumen terdekat
    closest_indices = np.argsort(distances[0])[:top_k]
    similarities = distances[0][closest_indices]

    return closest_indices, similarities

# Input teks dan preprocessing
query_text = input("Masukkan teks: ")
processed_query_text = preprocess_text(query_text)

# Melakukan pencarian dengan teks yang telah dipreproses
top_k = 10
indices, distances = search_similar_documents(processed_query_text, top_k)

# Menampilkan hasil pencarian
print("\nHasil pencarian untuk query:", processed_query_text)
for idx, distance in zip(indices, distances):
    print(f"Dokumen ke-{idx + 1}: {documents[idx]}")
    print(f"Jarak Euclidean: {distance:.4f}\n")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Masukkan teks: stasiun

Hasil pencarian untuk query: stasiun
Dokumen ke-65: jakarta krl rangkasbitung menghubungkan stasiun rangkasbitung banten stasiun tanah abang jakarta jalur transportasi pekerja rute memfasilitasi mobilitas masyarakat bepergian cepat kemacetan jalan raya bepergian stasiun rangkasbitung traveler stasiun dilewati jadwal keberangkatannya ketahui rute perjalanan transit stasiun tanah abang stasiun manggarai stasiun duri rute krl rangkasbitung transit jadwal keberangkatannya rute krl jabodetabek terbaru foto commuterlineid rute commuter line rangkasbitung melayani lintasan tujuan stasiun tanah abang stasiun merak stasiunstasiun rangkasbitungtanah abang traveler tanah abang stasiun jadwal keberangkatannya kereta berangkat rangkasbitung stasiun tanah abang laman commuter line jadwal keberangkatannya rutenya rangkasbitung citeras maja cikoya tigaraksa tenjo daru cilejit parung cicayur cisauk serpong rawa buntu sudimara jurang mangu pondok ranji kebayoran palmerah tanah ab

Nilai jarak ini menunjukkan tingkat kemiripan: semakin kecil nilai jaraknya, semakin relevan dokumen tersebut dengan kata kunci. Dengan jarak 0.8262, dokumen ini dianggap cukup relevan dalam konteks pencarian.

**Cleaning:**

*   re.sub(r'\d+', '', text): Menghapus semua angka dari teks menggunakan ekspresi reguler.

*   text.translate(...): Menghapus semua tanda baca dengan menggunakan metode translate.

*   text.lower(): Mengubah semua huruf dalam teks menjadi huruf kecil untuk menghindari perbedaan antara huruf besar dan kecil saat analisis.

*   word_tokenize(text): Memisahkan teks menjadi kata-kata (token).

**Stopword Removal:**

*   stop_words = set(stopwords.words("indonesian")): Mengambil daftar kata-kata umum yang tidak memiliki makna penting dalam konteks analisis.

*   words = [word for word in words if word not in stop_words]: Menggunakan pemahaman daftar untuk menghapus semua kata yang terdapat dalam daftar stopwords.

*   Return: Fungsi mengembalikan string yang terdiri dari kata-kata yang telah diproses, digabungkan kembali menjadi satu kalimat.

**Preprocessing:**

*   query_text = preprocess_text(query_text): Memanggil fungsi preprocess_text untuk membersihkan dan mempersiapkan teks query sebelum analisis.

**Transformasi:**

*   query_tfidf = tfidf_vectorizer.transform([query_text]): Mengubah teks query yang telah diproses menjadi representasi TF-IDF.

*   query_svd = svd.transform(query_tfidf): Mengubah matriks TF-IDF dari query menjadi bentuk yang direduksi dimensinya menggunakan SVD.

**Menghitung Jarak:**

*   distances = euclidean_distances(query_svd, svd_matrix): Menghitung jarak Euclidean antara representasi SVD dari query dan semua dokumen yang ada dalam matriks SVD.

**Mengambil Dokumen Terdekat:**

*   closest_indices = np.argsort(distances[0])[:top_k]: Mengurutkan jarak dari yang terkecil dan mengambil indeks dari top_k dokumen terdekat.

*   similarities = distances[0][closest_indices]: Mengambil jarak dari dokumen terdekat sesuai dengan indeks yang diperoleh.

**Melakukan Pencarian**

*   top_k = 5 : Variabel ini menetapkan jumlah maksimum dokumen yang ingin ditampilkan sebagai hasil pencarian. Dalam hal ini, hanya 5 dokumen terdekat yang akan diambil berdasarkan kesamaan dengan query.

*   indices, distances = search_similar_documents(processed_query_text, top_k):

*   Fungsi search_similar_documents dipanggil dengan argumen processed_query_text dan top_k.

**Menampilkan Hasil Pencarian**

*   print("\nHasil pencarian untuk query:", processed_query_text):

*   print("\nHasil pencarian untuk query:", processed_query_text):

*   for idx, distance in zip(indices, distances):

*   Menggunakan loop untuk iterasi melalui indices (indeks dokumen terdekat) dan distances (jarak Euclidean untuk setiap dokumen).

*   Fungsi zip digunakan untuk menggabungkan dua list ini sehingga kita dapat mengakses elemen yang berpasangan.

*   print(f"Dokumen ke-{idx + 1}: {documents[idx]}"):

*   Menampilkan informasi tentang dokumen terdekat yang ditemukan, termasuk urutan dokumen (dengan menambahkan 1 pada indeks) dan isi dokumen dari list documents.

*   print(f"Jarak Euclidean: {distance:.4f}\n"):

*   Menampilkan jarak Euclidean antara query dan dokumen tersebut, dengan format hingga 4 angka desimal.



























In [15]:
import re
import string
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import nltk
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import euclidean_distances

# Mengunduh stopwords dan tokenizer
nltk.download('stopwords')
nltk.download('punkt')

# Contoh DataFrame news_data (Anda harus menggantinya dengan DataFrame Anda sendiri)
# news_data = pd.DataFrame({
#     'nomor': [...], # Ensure this column exists in your DataFrame
#     'tanggal': [...],
#     'judul': [...],
#     'kategori': [...],
#     'isi': [...],
#     'stopword_removal': [...]
# })

# ... (rest of the code remains the same) ...

# Menampilkan hasil pencarian
print("\nHasil pencarian untuk query:", processed_query_text)
for idx, distance in zip(indices, distances):
    # Ambil informasi dari news_data, using correct column names if different
    # If 'nomor' column doesn't exist, replace with actual column name
    # For example, if the column with document number is named 'id', use:
    # nomor_berita = news_data['id'].iloc[idx]
    nomor_berita = news_data.iloc[idx].get('nomor', idx + 1)  # Get 'nomor' if exists, else default to idx + 1
    tanggal_berita = news_data['tanggal'].iloc[idx]
    judul_berita = news_data['judul'].iloc[idx]
    kategori_berita = news_data['kategori'].iloc[idx]
    isi_berita = news_data['isi'].iloc[idx]

    # Menampilkan informasi berita
    print(f"Dokumen ke-{nomor_berita}:") # Output using retrieved or default nomor_berita
    print(f"Tanggal: {tanggal_berita}")
    print(f"Judul: {judul_berita}")
    print(f"Kategori: {kategori_berita}")
    print(f"Isi Berita: {isi_berita}")
    print(f"Jarak Euclidean: {distance:.4f}\n")


Hasil pencarian untuk query: stasiun
Dokumen ke-65:
Tanggal: Rabu, 16 Okt 2024 22:10 WIB
Judul: Rute KRL Rangkasbitung, Cek Dulu Jadwal Keberangkatannya
Kategori: Pariwisata
Isi Berita: Jakarta - KRL Rangkasbitung menghubungkan Stasiun Rangkasbitung yang ada di Banten dengan stasiun lainnya, termasuk Tanah Abang di Jakarta. Selain menjadi jalur transportasi bagi pekerja, rute ini juga memfasilitasi mobilitas masyarakat yang ingin bepergian dengan cepat tanpa kemacetan di jalan raya. Nah, untuk bepergian dari Stasiun Rangkasbitung, traveler perlu mengetahui stasiun apa saja yang dilewati serta jadwal keberangkatannya. Ketahui juga rute perjalanan lainnya dengan transit di Stasiun Tanah Abang, Stasiun Manggarai, dan Stasiun Duri. Rute KRL Rangkasbitung Tanpa Transit dan Jadwal Keberangkatannya Rute KRL Jabodetabek Terbaru. Foto: Commuterline.id Rute Commuter Line Rangkasbitung melayani lintasan dengan tujuan akhir Stasiun Tanah Abang dan Stasiun Merak. Berikut stasiun-stasiun yang akan 

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
